In [1]:
pip install rdkit pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

In [2]:
def generate_morgan_features(input_path, output_path, radius=2, nbits=512):

   
    df = pd.read_csv(input_path, sep=';')
    
    unique_mols = df[['item', 'SMILES']].drop_duplicates().dropna(subset=['SMILES'])
    
    
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nbits)
    
    fp_records = []
    for _, row in unique_mols.iterrows():
        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol:

            # Generate the bit vector as a list of integers
            bits = list(generator.GetFingerprintAsNumPy(mol))
            
            # Map each bit to a column name like morgan_0, morgan_1, ..., morgan_511
            feature_dict = {f'morgan_{i}': val for i, val in enumerate(bits)}
            feature_dict['item'] = row['item']
            fp_records.append(feature_dict)
    
   
    fp_df = pd.DataFrame(fp_records)
    enriched_df = df.merge(fp_df, on='item', how='left')
    
    
    enriched_df.to_csv(output_path, index=False)
    print(f"File saved: {output_path} ({len(unique_mols)} substances processed)")

In [3]:
generate_morgan_features(
        input_path='7Q27.signatures_with_smiles.csv',
        output_path='signatures_with_morgan.csv'
    )

File saved: signatures_with_morgan.csv (6 substances processed)
